## Chapter 7: Parameters and Multivariate Generating Functions 

This is the Sage notebook for [**Chapter 7 of enumeration.ca**](https://enumeration.ca/extensions/parameters/) by Stephen Melczer. We examine how to use lazy power series to study bivariate generating functions in Sage.

In [1]:
# Recall that we can use lazy power series to encode GFs of classes
S.<x> = LazyPowerSeriesRing(QQ)
B = S.undefined()
B.define(1+x*B^2)
B[:10]

[1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862]

In [2]:
# Specifications with tracking parameters work similarly
# Here we enumerate binary trees by size and number of leaves
# Recall from the notes that we remove the empty tree to more easily track leaves
S.<x, u> = LazyPowerSeriesRing(QQ)
B = S.undefined()
B.define(x*u + 2*x*B + x*B^2)
B

x*u + 2*x^2*u + 4*x^3*u + (8*x^4*u+x^3*u^2) + (16*x^5*u+6*x^4*u^2) + O(x,u)^7

In [3]:
# NOTE: This expands as a series with degree given by x AND u
B[10]

256*x^9*u + 672*x^8*u^2 + 120*x^7*u^3

In [4]:
# It's usually more natural to define series in x whose coefficients are polynomials in the other variables
S.<u> = QQ['u']
T.<x> = LazyPowerSeriesRing(S)
B = T.undefined()
B.define(x*u + 2*x*B + x*B^2)
B

u*x + 2*u*x^2 + ((u^2+4*u)*x^3) + ((6*u^2+8*u)*x^4) + ((2*u^3+24*u^2+16*u)*x^5) + ((20*u^3+80*u^2+32*u)*x^6) + O(x^7)

In [5]:
# Setting u=1 recovers the univariate GF above (except for the constant term)
[p.subs(u=1) for p in B[:10]]

[1, 2, 5, 14, 42, 132, 429, 1430, 4862]

In [6]:
# There are four trees with size 4 and one leaf, and one tree with size 4 and two leaves
print(B[3])

u^2 + 4*u


In [7]:
# Print out the trees and their leaf counts
def count_leaves(t): return str(t).count('[., .]')
BT = BinaryTrees(3)
print(ascii_art(list(BT)))
[count_leaves(k) for k in BT]

[ o    , o  ,   o  ,   o,     o ]
[  \      \    / \    /      /  ]
[   o      o  o   o  o      o   ]
[    \    /           \    /    ]
[     o  o             o  o     ]


[1, 1, 2, 1, 1]

In [8]:
# Generate all trees with size 10 and count how many have k leaves for 0 <= k <= 5 (the max)
from collections import Counter
[Counter([count_leaves(k) for k in BinaryTrees(10)])[k] for k in range(6)]

[0, 512, 4608, 8064, 3360, 252]

In [9]:
# This matches the GF -- except we compute the GF almost instantly
B[10]

252*u^5 + 3360*u^4 + 8064*u^3 + 4608*u^2 + 512*u

In [10]:
# We can go quite far
B[100]

100891344545564193334812497256*u^50 + 164789196091088182446860412184800*u^49 + 77516837841247881023003137891729920*u^48 + 16655046301890973294085245627023114240*u^47 + 2000456116927126903434016724756887388160*u^46 + 150579787710514643276669622554427523399680*u^45 + 7644819991456897274046303914301705034137600*u^44 + 275504750930218088428487752492549065230254080*u^43 + 7317082061470203936791895308846229585379983360*u^42 + 147368600115224458235738523062376694105547735040*u^41 + 2301757182752077252443915979259978841267602718720*u^40 + 28385305969116525800889398637514363576106405068800*u^39 + 280446822974871274912787258538641912131931282079744*u^38 + 2246770558989567022948027837637210988361797051875328*u^37 + 14742356574256666377176222067649088849743417108856832*u^36 + 79894061434681288753729203463388610540544970138320896*u^35 + 360128534497237627336885424702395630845638312365916160*u^34 + 1358199044389581908813396458877606379189264492351455232*u^33 + 430708165428047596308392390562988689616

In [11]:
# Remember that the average number of leaves approaches n/4 as n -> infinity
(B[100].diff()(1) / B[100](1)).n()

25.3768844221106

In [12]:
# We can also define series implicitly
S.<x, y> = LazyPowerSeriesRing(QQ)
D = S.undefined()

# Let D(x,y) be the GF of Dyck paths tracking length (x) and height (y)
# [A path] is [empty]  or [up step]  or [down step] - [down step on x-axis]
# So D     =  1        +  x*y*D(x,y) +  x/y*D(x,y)  - x/y*D(x,0)
S.define_implicitly([D], [y + x*y^2*D + x*(D - D(x, 0)) - y*D])
D

1 + (x^2+x*y) + (2*x^4+2*x^3*y+x^2*y^2) + (5*x^6+5*x^5*y+3*x^4*y^2+x^3*y^3) + O(x,y)^7

In [13]:
# Note in this case the series is computed in both variables
# We can reconstruct any coefficient of x we want by computing coefficients up to twice the desired size
N = 10
add([D[n][N,n-N]*y^n for n in range(N,2*N+1)])

42*y^10 + 90*y^12 + 75*y^14 + 35*y^16 + 9*y^18 + y^20